# Diagnóstico inicial de calidad de datos
**Proyecto:** Barreras para la actividad física – PulseUp
**Evidencia:** GA1-220501094-01-AA1-EV05 · **Autor:** Santiago Varela Peña · ADSO, Ficha 3223873

**Pregunta analítica:** ¿Qué barreras impiden a los adultos realizar actividad física de forma regular, qué factores
sociodemográficos y de salud se asocian con la inactividad, y cuáles de esas barreras puede abordar una aplicación móvil
de hábitos saludables como PulseUp?

**Objetivo de este notebook:** cargar, inspeccionar y diagnosticar la calidad de los tres datasets del proyecto
antes de limpiarlos o analizarlos. En esta etapa **no se modifican los datos**: solo se describen y se documentan los problemas.

## Momento 1. Reconocimiento de los datasets

| # | Dataset | Fuente | Formato | Relación con la pregunta |
|---|---|---|---|---|
| 1 | BRFSS 2023 | CDC de EE. UU. | SAS Transport (`.XPT`) | Factores sociodemográficos y de salud asociados a la inactividad |
| 2 | Actividad física en Bogotá D.C. | Secretaría Distrital de Salud (Datos Abiertos Colombia) | CSV | Contexto local de la inactividad |
| 3 | Eurobarómetro 97.3 (ZA7888) | Comisión Europea / GESIS | Stata (`.dta`) o SPSS (`.sav`) | Barreras declaradas para no hacer actividad física |

**Aspectos de calidad a revisar:** valores faltantes, tipos de datos, códigos especiales
(no sabe / no responde), valores fuera de rango, duplicados y limitaciones para responder la pregunta.

## Momento 2. Preparación y carga de datos

In [1]:
import re
import warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*select_dtypes.*")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# En Google Colab los datos se leen desde Google Drive; en local, desde data/raw/
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_RAW = Path("/content/drive/MyDrive/pulseup-datasets")
except ModuleNotFoundError:
    DATA_RAW = Path("..") / "data" / "raw"

print("Pandas:", pd.__version__)
print("Datos en:", DATA_RAW)
print("Archivos:", sorted(p.name for p in DATA_RAW.iterdir() if p.is_file()))

Pandas: 3.1.1
Datos en: ..\data\raw
Archivos: ['.gitkeep', 'LLCP2023.XPT', 'README.md', 'ZA7888_v1-0-0.dta', 'actividad_fisica_bogota.csv']


In [2]:
def buscar_archivo(*patrones):
    """Devuelve el primer archivo de la carpeta de datos que coincida con alguno de los patrones."""
    for patron in patrones:
        encontrados = sorted(DATA_RAW.glob(patron))
        if encontrados:
            return encontrados[0]
    raise FileNotFoundError(
        f"No se encontró ningún archivo con {patrones} en {DATA_RAW.resolve()}. "
        "Revisa data/raw/README.md para descargarlo."
    )

### Dataset 1: BRFSS 2023
El archivo completo tiene más de 400 000 filas y más de 300 columnas. Para no saturar la memoria,
se lee por bloques y se conservan solo las variables relacionadas con la pregunta analítica.

In [3]:
VARIABLES_BRFSS = [
    "_STATE",     # estado de residencia
    "SEXVAR",     # sexo
    "_AGEG5YR",   # grupo de edad (5 años)
    "_AGE80",     # edad imputada (máx. 80)
    "EDUCA",      # nivel educativo
    "EMPLOY1",    # situación laboral
    "INCOME3",    # ingresos del hogar
    "EXERANY2",   # actividad física en los últimos 30 días
    "_TOTINDA",   # indicador calculado de actividad física
    "GENHLTH",    # salud general percibida
    "SLEPTIM1",   # horas de sueño
    "_BMI5",      # índice de masa corporal (2 decimales implícitos)
    "ADDEPEV3",   # diagnóstico de trastorno depresivo
    "_LLCPWT",    # peso muestral
]

ruta_brfss = buscar_archivo("LLCP2023*", "*.XPT*", "*.xpt")
bloques = []
with pd.read_sas(ruta_brfss, format="xport", chunksize=50_000, encoding="latin-1") as lector:
    for bloque in lector:
        columnas = [c for c in VARIABLES_BRFSS if c in bloque.columns]
        bloques.append(bloque[columnas])
df_brfss = pd.concat(bloques, ignore_index=True)

faltan = [c for c in VARIABLES_BRFSS if c not in df_brfss.columns]
print(f"BRFSS cargado: {df_brfss.shape[0]:,} filas x {df_brfss.shape[1]} columnas")
print("Variables no encontradas en el archivo:", faltan or "ninguna")

BRFSS cargado: 433,323 filas x 13 columnas
Variables no encontradas en el archivo: ['SLEPTIM1']


### Dataset 2: Actividad física en Bogotá D.C.

In [4]:
ruta_bogota = buscar_archivo("*actividad*fisica*.csv", "*.csv", "*.xlsx")
if ruta_bogota.suffix == ".xlsx":
    df_bogota = pd.read_excel(ruta_bogota)
else:
    # El archivo publicado por la Secretaría Distrital de Salud viene en latin-1 (ISO-8859-1)
    # y usa ";" como separador. La coma decimal ("36,7") se deja sin convertir para diagnosticarla.
    df_bogota = pd.read_csv(ruta_bogota, encoding="latin-1", sep=";")
print(f"Bogotá cargado: {df_bogota.shape[0]:,} filas x {df_bogota.shape[1]} columnas")

Bogotá cargado: 168 filas x 4 columnas


### Dataset 3: Eurobarómetro 97.3 (ZA7888)
Se intenta conservar las etiquetas de las respuestas. Si el archivo trae etiquetas duplicadas,
se carga con los códigos numéricos.

**Nota:** este dataset requiere una cuenta de GESIS y no se puede descargar automáticamente. Si el archivo aún no está en `data/raw/`, el notebook continúa con los otros dos datasets y las celdas del Eurobarómetro muestran un aviso en lugar de fallar.

In [5]:
import pyreadstat

try:
    ruta_euro = buscar_archivo("ZA7888*.dta", "ZA7888*.sav")
except FileNotFoundError as e:
    ruta_euro = None
    df_eurobarometro = None
    print("AVISO:", e)

if ruta_euro is not None:
    if ruta_euro.suffix == ".dta":
        # Se cargan los códigos numéricos: la variable d9b tiene etiquetas duplicadas que impiden convertirlas
        # a categorías. El significado de cada código se toma de los metadatos (pyreadstat, más abajo).
        df_eurobarometro = pd.read_stata(ruta_euro, convert_categoricals=False)
        etiquetas = False
    else:
        df_eurobarometro = pd.read_spss(ruta_euro)
        etiquetas = True
    # Metadatos del archivo: texto de cada variable y significado de cada código numérico
    lector_meta = pyreadstat.read_dta if ruta_euro.suffix == ".dta" else pyreadstat.read_sav
    _, meta_euro = lector_meta(ruta_euro, metadataonly=True)
    etiquetas_variables = meta_euro.column_names_to_labels
    etiquetas_valores = meta_euro.variable_value_labels
    print(f"Eurobarómetro cargado: {df_eurobarometro.shape[0]:,} filas x {df_eurobarometro.shape[1]} columnas")
    print("Etiquetas de respuesta conservadas:", etiquetas)

Eurobarómetro cargado: 26,569 filas x 501 columnas
Etiquetas de respuesta conservadas: False


## Momento 3. Inspección de la estructura

### Dataset 1: BRFSS 2023

In [6]:
df_brfss.head()

,_STATE,SEXVAR,_AGEG5YR,_AGE80,EDUCA,EMPLOY1,INCOME3,EXERANY2,_TOTINDA,GENHLTH,_BMI5,ADDEPEV3,_LLCPWT
0,1.0,2.0,13.0,80.0,5.0,7.0,99.0,2.0,2.0,2.0,3047.0,2.0,605.427887
1,1.0,2.0,13.0,80.0,5.0,7.0,99.0,1.0,1.0,2.0,2856.0,1.0,1121.992705
2,1.0,2.0,13.0,80.0,4.0,7.0,2.0,1.0,1.0,4.0,2231.0,2.0,600.963308
3,1.0,2.0,12.0,78.0,5.0,7.0,99.0,1.0,1.0,2.0,2744.0,1.0,605.427887
4,1.0,2.0,12.0,76.0,5.0,8.0,7.0,1.0,1.0,4.0,2585.0,1.0,281.711042


In [7]:
df_brfss.tail()

,_STATE,SEXVAR,_AGEG5YR,_AGE80,EDUCA,EMPLOY1,INCOME3,EXERANY2,_TOTINDA,GENHLTH,_BMI5,ADDEPEV3,_LLCPWT
433318,78.0,1.0,10.0,65.0,5.0,1.0,5.0,1.0,1.0,3.0,2921.0,2.0,39.700749
433319,78.0,2.0,3.0,30.0,6.0,1.0,6.0,2.0,2.0,2.0,2496.0,2.0,39.132414
433320,78.0,2.0,7.0,51.0,6.0,1.0,10.0,1.0,1.0,2.0,3438.0,2.0,40.185536
433321,78.0,2.0,10.0,65.0,6.0,4.0,3.0,1.0,1.0,3.0,2386.0,2.0,1.737941
433322,78.0,1.0,6.0,48.0,2.0,2.0,2.0,1.0,1.0,5.0,1794.0,2.0,34.093637


In [8]:
df_brfss.shape

(433323, 13)

In [9]:
df_brfss.columns

Index(['_STATE', 'SEXVAR', '_AGEG5YR', '_AGE80', 'EDUCA', 'EMPLOY1', 'INCOME3', 'EXERANY2', '_TOTINDA', 'GENHLTH', '_BMI5', 'ADDEPEV3', '_LLCPWT'], dtype='str')

In [10]:
df_brfss.dtypes

_STATE      float64
SEXVAR      float64
_AGEG5YR    float64
_AGE80      float64
EDUCA       float64
EMPLOY1     float64
INCOME3     float64
EXERANY2    float64
_TOTINDA    float64
GENHLTH     float64
_BMI5       float64
ADDEPEV3    float64
_LLCPWT     float64
dtype: object

In [11]:
df_brfss.info()

<class 'pandas.DataFrame'>
RangeIndex: 433323 entries, 0 to 433322
Data columns (total 13 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   _STATE    433323 non-null  float64
 1   SEXVAR    433323 non-null  float64
 2   _AGEG5YR  433323 non-null  float64
 3   _AGE80    433323 non-null  float64
 4   EDUCA     433314 non-null  float64
 5   EMPLOY1   430355 non-null  float64
 6   INCOME3   425248 non-null  float64
 7   EXERANY2  433321 non-null  float64
 8   _TOTINDA  433323 non-null  float64
 9   GENHLTH   433319 non-null  float64
 10  _BMI5     392788 non-null  float64
 11  ADDEPEV3  433320 non-null  float64
 12  _LLCPWT   433323 non-null  float64
dtypes: float64(13)
memory usage: 43.0 MB


In [12]:
df_brfss.describe()

,_STATE,SEXVAR,_AGEG5YR,_AGE80,EDUCA,EMPLOY1,INCOME3,EXERANY2,_TOTINDA,GENHLTH,_BMI5,ADDEPEV3,_LLCPWT
count,433323.000000,433323.000000,433323.000000,433323.000000,433314.000000,430355.000000,425248.000000,433321.000000,433323.000000,433319.000000,392788.000000,433320.000000,433323.000000
mean,30.486898,1.529723,7.836318,55.375461,5.063409,3.921586,22.109995,1.265353,1.269667,2.629645,2847.928783,1.829006,586.490514
std,16.670560,0.499116,3.694292,17.909598,1.049580,2.892771,32.180148,0.548392,0.598848,1.081803,654.197490,0.595546,1190.719872
min,1.000000,1.000000,1.000000,18.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1202.000000,1.000000,0.159104
25%,18.000000,1.000000,5.000000,41.000000,4.000000,1.000000,6.000000,1.000000,1.000000,2.000000,2410.000000,2.000000,103.366483
50%,29.000000,2.000000,8.000000,58.000000,5.000000,2.000000,8.000000,1.000000,1.000000,3.000000,2740.000000,2.000000,258.271915
75%,45.000000,2.000000,11.000000,71.000000,6.000000,7.000000,10.000000,1.000000,1.000000,3.000000,3162.000000,2.000000,604.551174
max,78.000000,2.000000,14.000000,80.000000,9.000000,9.000000,99.000000,9.000000,9.000000,9.000000,9984.000000,9.000000,69785.820912


In [13]:
if df_brfss.select_dtypes(include=["object", "category"]).shape[1] > 0:
    display(df_brfss.describe(include="object") if df_brfss.select_dtypes(include="object").shape[1] else df_brfss.describe(include="category"))
else:
    print("Este dataset no tiene columnas de tipo object.")

Este dataset no tiene columnas de tipo object.


**Interpretación:** se cargaron **433 323 registros** y 13 de las 14 variables previstas: `SLEPTIM1` (horas de sueño) no existe en el archivo de 2023.
Todas las columnas son `float64`, incluso las categóricas (sexo, educación, ingresos, salud), porque el formato SAS guarda todo como número.
`describe()` muestra máximos de 9 y 99 en variables cuyas respuestas válidas llegan a 5 u 11: son códigos de "no sabe / no responde", no valores reales.
`_BMI5` va de 1 202 a 9 984 porque tiene dos decimales implícitos (IMC de 12,02 a 99,84), y el peso muestral `_LLCPWT` va de 0,16 a 69 785,82,
lo que confirma que los porcentajes poblacionales deben calcularse ponderados.

### Dataset 2: Bogotá

In [14]:
df_bogota.head()

,Año,Localidad,Percepción,Porcentaje
0,2017,Usaquén,No practicó deporte ni tuvo actividad física e...,"36,7"
1,2017,Chapinero,No practicó deporte ni tuvo actividad física e...,"30,5"
2,2017,Santa Fe,No practicó deporte ni tuvo actividad física e...,"46,1"
3,2017,San Cristóbal,No practicó deporte ni tuvo actividad física e...,"56,3"
4,2017,Usme,No practicó deporte ni tuvo actividad física e...,"57,5"


In [15]:
df_bogota.tail()

,Año,Localidad,Percepción,Porcentaje
163,2021,La Candelaria,3 o más veces por semana,"27,1"
164,2021,Rafael Uribe Uribe,3 o más veces por semana,"17,7"
165,2021,Ciudad Bolívar,3 o más veces por semana,"14,7"
166,2021,Sumapaz,3 o más veces por semana,"26,8"
167,2021,Bogotá D.C.,3 o más veces por semana,"20,4"


In [16]:
df_bogota.shape

(168, 4)

In [17]:
df_bogota.columns

Index(['Año', 'Localidad', 'Percepción', 'Porcentaje'], dtype='str')

In [18]:
df_bogota.dtypes

Año           int64
Localidad       str
Percepción      str
Porcentaje      str
dtype: object

In [19]:
df_bogota.info()

<class 'pandas.DataFrame'>
RangeIndex: 168 entries, 0 to 167
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   Año         168 non-null    int64
 1   Localidad   168 non-null    str  
 2   Percepción  168 non-null    str  
 3   Porcentaje  168 non-null    str  
dtypes: int64(1), str(3)
memory usage: 13.0 KB


In [20]:
df_bogota.describe()

,Año
count,168.000000
mean,2019.000000
std,2.005979
min,2017.000000
25%,2017.000000
50%,2019.000000
75%,2021.000000
max,2021.000000


In [21]:
if df_bogota.select_dtypes(include=["object", "category"]).shape[1] > 0:
    display(df_bogota.describe(include="object") if df_bogota.select_dtypes(include="object").shape[1] else df_bogota.describe(include="category"))
else:
    print("Este dataset no tiene columnas de tipo object.")

,Localidad,Percepción,Porcentaje
count,168,168,168
unique,21,4,134
top,Usaquén,No practicó deporte ni tuvo actividad física e...,"8,1"
freq,8,42,3


**Interpretación:** el archivo tiene **168 filas y 4 columnas, sin valores faltantes**. No son microdatos sino porcentajes agregados:
21 localidades (las 20 de Bogotá más el total "Bogotá D.C.") × 4 categorías de frecuencia (`Percepción`) × 2 años (2017 y 2021).
`Porcentaje` se lee como texto (`str`) porque usa coma decimal ("36,7"), y el archivo viene en latin-1 separado por punto y coma.

### Dataset 3: Eurobarómetro 97.3

In [22]:
if df_eurobarometro is not None:
    display(df_eurobarometro.head())
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

,studyno1,studyno2,doi,version,edition,survey,respondent_serial,serialid,uniqid,tnscntry,country,isocntry,q1_1,q1_2,q1_3,q1_4,q1_5,q1_6,q1_7,q1_8,q1_9,q1_10,q1_11,q1_12,q1_13,...,region_luxembourg,region_malta,region_netherlands,region_poland,region_portugal,region_romania,region_slovenia,region_slovakia,region_spain,region_sweden,w1,w1de,w5,w6,w7,w9,w10,w11,w85,w86,w13,w94,w14,w22,w92
0,7888,7888,doi:10.4232/1.14055,1.0.0 (2022-12-21),1,973,6,1,20000001,2,18,AT,0,0,0,0,0,0,0,0,0,0,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.746515,0.746515,NaN,NaN,NaN,NaN,0.830778,0.274903,0.353595,NaN,NaN,NaN,0.373504,0.383012,0.394492
1,7888,7888,doi:10.4232/1.14055,1.0.0 (2022-12-21),1,973,12,2,20000002,2,18,AT,0,0,0,0,0,0,0,0,0,0,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.746282,0.746282,NaN,NaN,NaN,NaN,0.830519,0.274817,0.353484,NaN,NaN,NaN,0.373387,0.382892,0.394369
2,7888,7888,doi:10.4232/1.14055,1.0.0 (2022-12-21),1,973,18,3,20000003,2,18,AT,0,0,0,0,0,0,0,0,0,0,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.642434,0.642434,NaN,NaN,NaN,NaN,0.714949,0.236575,0.304296,NaN,NaN,NaN,0.321429,0.329612,0.339491
3,7888,7888,doi:10.4232/1.14055,1.0.0 (2022-12-21),1,973,23,4,20000004,2,18,AT,0,0,0,0,0,0,0,0,0,0,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.812821,0.812821,NaN,NaN,NaN,NaN,0.904568,0.299320,0.385001,NaN,NaN,NaN,0.406679,0.417031,0.429531
4,7888,7888,doi:10.4232/1.14055,1.0.0 (2022-12-21),1,973,27,5,20000005,2,18,AT,0,0,0,0,0,0,0,0,0,0,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.609060,0.609060,NaN,NaN,NaN,NaN,0.677808,0.224285,0.288488,NaN,NaN,NaN,0.304731,0.312488,0.321855


In [23]:
if df_eurobarometro is not None:
    display(df_eurobarometro.tail())
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

,studyno1,studyno2,doi,version,edition,survey,respondent_serial,serialid,uniqid,tnscntry,country,isocntry,q1_1,q1_2,q1_3,q1_4,q1_5,q1_6,q1_7,q1_8,q1_9,q1_10,q1_11,q1_12,q1_13,...,region_luxembourg,region_malta,region_netherlands,region_poland,region_portugal,region_romania,region_slovenia,region_slovakia,region_spain,region_sweden,w1,w1de,w5,w6,w7,w9,w10,w11,w85,w86,w13,w94,w14,w22,w92
26564,7888,7888,doi:10.4232/1.14055,1.0.0 (2022-12-21),1,973,5155,26574,150026574,15,16,FI,0,0,0,0,0,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.038558,1.038558,NaN,NaN,NaN,NaN,0.708714,0.234512,0.301641,NaN,NaN,NaN,0.318626,0.326738,0.336529
26565,7888,7888,doi:10.4232/1.14055,1.0.0 (2022-12-21),1,973,5166,26575,280026575,28,25,MT,0,0,0,0,0,0,0,0,0,0,0,0,0,...,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.996886,1.996886,NaN,NaN,NaN,NaN,NaN,NaN,0.109407,NaN,0.260703,0.246505,0.115569,0.118513,0.122058
26566,7888,7888,doi:10.4232/1.14055,1.0.0 (2022-12-21),1,973,5178,26576,150026576,15,16,FI,0,0,0,0,0,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.808824,2.808824,NaN,NaN,NaN,NaN,1.916748,0.634247,0.815802,NaN,NaN,NaN,0.861737,0.883678,0.910156
26567,7888,7888,doi:10.4232/1.14055,1.0.0 (2022-12-21),1,973,5195,26577,150026577,15,16,FI,0,0,0,0,0,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.845292,0.845292,NaN,NaN,NaN,NaN,0.576829,0.190871,0.245509,NaN,NaN,NaN,0.259332,0.265935,0.273904
26568,7888,7888,doi:10.4232/1.14055,1.0.0 (2022-12-21),1,973,5196,26578,150026578,15,16,FI,0,0,0,0,0,0,0,0,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.036574,1.036574,NaN,NaN,NaN,NaN,0.707361,0.234064,0.301065,NaN,NaN,NaN,0.318017,0.326114,0.335886


In [24]:
if df_eurobarometro is not None:
    print(df_eurobarometro.shape)
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

(26569, 501)


In [25]:
if df_eurobarometro is not None:
    print(df_eurobarometro.columns)
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

Index(['studyno1', 'studyno2', 'doi', 'version', 'edition', 'survey', 'respondent_serial', 'serialid', 'uniqid', 'tnscntry',
       ...
       'w9', 'w10', 'w11', 'w85', 'w86', 'w13', 'w94', 'w14', 'w22', 'w92'],
      dtype='str', length=501)


In [26]:
if df_eurobarometro is not None:
    display(df_eurobarometro.dtypes)
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

studyno1      int16
studyno2      int16
doi             str
version         str
edition        int8
             ...   
w13         float64
w94         float64
w14         float64
w22         float64
w92         float64
Length: 501, dtype: object

In [27]:
if df_eurobarometro is not None:
    df_eurobarometro.info()
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

<class 'pandas.DataFrame'>
RangeIndex: 26569 entries, 0 to 26568
Columns: 501 entries, studyno1 to w92
dtypes: float64(166), int16(8), int32(2), int8(322), str(3)
memory usage: 44.0 MB


In [28]:
if df_eurobarometro is not None:
    display(df_eurobarometro.describe())
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

,studyno1,studyno2,edition,survey,respondent_serial,serialid,uniqid,tnscntry,country,q1_1,q1_2,q1_3,q1_4,q1_5,q1_6,q1_7,q1_8,q1_9,q1_10,q1_11,q1_12,q1_13,q1_14,q1_15,q1_16,...,region_luxembourg,region_malta,region_netherlands,region_poland,region_portugal,region_romania,region_slovenia,region_slovakia,region_spain,region_sweden,w1,w1de,w5,w6,w7,w9,w10,w11,w85,w86,w13,w94,w14,w22,w92
count,26569.0,26569.0,26569.0,26569.0,26569.000000,26569.000000,2.656900e+04,26569.000000,26569.000000,26569.000000,26569.000000,26569.000000,26569.000000,26569.000000,26569.000000,26569.000000,26569.000000,26569.000000,26569.000000,26569.000000,26569.0,26569.000000,26569.000000,26569.000000,26569.000000,...,502.000000,504.0,1032.000000,1013.000000,1005.000000,1057.000000,1022.000000,1010.000000,1006.000000,1038.000000,26569.000000,26569.000000,5682.000000,7697.000000,8711.000000,11218.000000,3052.000000,14270.000000,18307.000000,8262.000000,9195.000000,12299.000000,23465.000000,25561.000000,26569.000000
mean,7888.0,7888.0,1.0,973.0,1769.028680,13287.853137,2.095812e+08,20.956792,16.567541,0.040724,0.037751,0.056796,0.039181,0.037224,0.040875,0.035944,0.040837,0.011404,0.039482,0.041213,0.0,0.037638,0.039369,0.038504,0.018254,...,2.601594,1.0,7.464147,7.269497,2.590050,4.240303,6.625245,2.544554,10.485089,1.815029,1.000034,0.999996,1.035296,1.026363,1.023285,1.000227,1.000000,1.000135,1.000137,1.000066,1.000185,1.000144,1.000123,1.000107,1.000096
std,0.0,0.0,0.0,0.0,1614.011623,7672.284592,1.142877e+08,11.428358,9.470940,0.197654,0.190596,0.231456,0.194029,0.189313,0.198004,0.186154,0.197916,0.106182,0.194743,0.198787,0.0,0.190322,0.194475,0.192412,0.133872,...,1.311943,0.0,3.039774,4.746272,1.395461,2.183132,3.616193,1.049090,4.463815,0.769780,0.411746,0.415390,0.908435,1.124924,1.163744,1.023105,0.507585,1.130224,1.386331,0.941129,1.519644,1.320062,1.345948,1.330630,1.352869
min,7888.0,7888.0,1.0,973.0,1.000000,1.000000,2.000000e+07,2.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.192222,0.192222,0.015569,0.019991,0.021555,0.020788,0.301223,0.024569,0.021747,0.137901,0.051820,0.048998,0.022972,0.023557,0.024262
25%,7888.0,7888.0,1.0,973.0,673.000000,6644.000000,1.200258e+08,12.000000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.0,5.000000,3.000000,1.000000,2.000000,2.000000,2.000000,8.000000,1.000000,0.732947,0.729893,0.273840,0.182764,0.216437,0.255967,0.668809,0.266834,0.123673,0.463473,0.204331,0.223773,0.183437,0.214961,0.208127
50%,7888.0,7888.0,1.0,973.0,1360.000000,13288.000000,1.900130e+08,19.000000,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,3.000000,1.0,8.000000,6.000000,3.000000,4.000000,8.000000,2.000000,12.000000,2.000000,0.921186,0.923678,0.927231,0.471192,0.447085,0.497240,0.878570,0.444772,0.387780,0.674438,0.384495,0.509573,0.452485,0.476594,0.467612
75%,7888.0,7888.0,1.0,973.0,2276.000000,19932.000000,3.200190e+08,32.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,...,4.000000,1.0,10.000000,11.000000,4.000000,6.000000,9.000000,4.000000,15.000000,2.000000,1.173823,1.178844,1.582892,1.727585,1.729777,1.633607,1.215716,1.668660,1.258101,1.175388,1.120086,1.221017,1.095879,1.022822,1.006782
max,7888.0,7888.0,1.0,973.0,8727.000000,26578.000000,4.200055e+08,42.000000,32.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,1.000000,1.000000,1.000000,1.000000,...,4.000000,1.0,12.000000,16.000000,5.000000,8.000000,12.000000,4.000000,17.000000,3.000000,6.5840

In [29]:
if df_eurobarometro is not None:
    if df_eurobarometro.select_dtypes(include=["object", "category"]).shape[1] > 0:
        display(df_eurobarometro.describe(include="object") if df_eurobarometro.select_dtypes(include="object").shape[1] else df_eurobarometro.describe(include="category"))
    else:
        print("Este dataset no tiene columnas de tipo object.")
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

,doi,version,isocntry
count,26569,26569,26569
unique,1,1,28
top,doi:10.4232/1.14055,1.0.0 (2022-12-21),BE
freq,26569,26569,1101


**Interpretación:** el archivo tiene **26 569 encuestados y 501 columnas**: identificadores, todas las preguntas de la encuesta (no solo el módulo de deporte),
variables sociodemográficas, regiones por país y factores de ponderación (`w1`, `w92`, etc.). Las respuestas se cargaron como códigos numéricos
(322 columnas `int8`) porque la variable `d9b` tiene etiquetas duplicadas que impiden convertirlas en categorías; su significado se recupera de los
metadatos con `pyreadstat`. `isocntry` tiene 28 códigos para los 27 países de la UE, por lo que hay que verificar si algún país está dividido.
Para el proyecto solo se necesitan `qb1` (frecuencia de práctica), `qb14_1` a `qb14_13` (barreras) y algunas variables sociodemográficas.

### Tabla de diagnóstico estructural

In [30]:
def resumen_tipos(df):
    conteo = df.dtypes.astype(str).value_counts()
    return ", ".join(f"{tipo}: {n}" for tipo, n in conteo.items())

observaciones = {
    "BRFSS 2023": "Subconjunto de 13 de las 350 variables del archivo (SLEPTIM1 no existe en 2023). "
                  "Todas las variables, incluso las categóricas, se leen como float64; usan códigos 7/9, 77/99 y 14 "
                  "para 'no sabe/no responde' y _BMI5 tiene 2 decimales implícitos.",
    "Bogotá": "Datos agregados (porcentajes), no microdatos: una fila por año, localidad y categoría de frecuencia. "
              "Archivo en latin-1 separado por ';'; 'Porcentaje' es texto con coma decimal.",
    "Eurobarómetro 97.3": "Microdatos por encuestado. Las respuestas se cargaron como códigos numéricos porque la "
                          "variable d9b tiene etiquetas duplicadas; muchas columnas region_* solo aplican a un país.",
}

tabla_estructural = pd.DataFrame([
    {"Dataset": nombre, "Filas": df.shape[0], "Columnas": df.shape[1], "Tipos de datos": resumen_tipos(df),
     "Observaciones": observaciones[nombre]}
    for nombre, df in [("BRFSS 2023", df_brfss), ("Bogotá", df_bogota), ("Eurobarómetro 97.3", df_eurobarometro)]
    if df is not None
])
with pd.option_context("display.max_colwidth", None):
    display(tabla_estructural)


,Dataset,Filas,Columnas,Tipos de datos,Observaciones
0,BRFSS 2023,433323,13,float64: 13,"Subconjunto de 13 de las 350 variables del archivo (SLEPTIM1 no existe en 2023). Todas las variables, incluso las categóricas, se leen como float64; usan códigos 7/9, 77/99 y 14 para 'no sabe/no responde' y _BMI5 tiene 2 decimales implícitos."
1,Bogotá,168,4,"str: 3, int64: 1","Datos agregados (porcentajes), no microdatos: una fila por año, localidad y categoría de frecuencia. Archivo en latin-1 separado por ';'; 'Porcentaje' es texto con coma decimal."
2,Eurobarómetro 97.3,26569,501,"int8: 322, float64: 166, int16: 8, str: 3, int32: 2",Microdatos por encuestado. Las respuestas se cargaron como códigos numéricos porque la variable d9b tiene etiquetas duplicadas; muchas columnas region_* solo aplican a un país.


## Momento 4. Diagnóstico de calidad

In [31]:
def resumen_faltantes(df):
    """Columnas con valores faltantes, ordenadas de mayor a menor."""
    faltantes = df.isna().sum()
    tabla = pd.DataFrame({"faltantes": faltantes, "porcentaje": (faltantes / len(df) * 100).round(2)})
    return tabla[tabla["faltantes"] > 0].sort_values("faltantes", ascending=False)


def texto_que_parece_numero(df):
    """Columnas de texto en las que al menos la mitad de los valores son números escritos como texto."""
    sospechosas = {}
    for col in df.select_dtypes(include=["object"]).columns:
        valores = df[col].dropna().astype(str).str.replace(",", ".", regex=False).str.strip()
        proporcion = pd.to_numeric(valores, errors="coerce").notna().mean() if len(valores) else 0
        if proporcion >= 0.5:
            sospechosas[col] = f"{proporcion:.0%} numéricos"
    return sospechosas

### Dataset 1: BRFSS 2023

In [32]:
print("1. Valores faltantes")
display(resumen_faltantes(df_brfss))

print("2. Tipos de datos")
print(df_brfss.dtypes.value_counts())

# El subconjunto no incluye un identificador del encuestado, así que dos personas distintas
# con las mismas 13 respuestas cuentan como "duplicadas".
print("3. Filas duplicadas (en las 13 variables seleccionadas):", df_brfss.duplicated().sum())

1. Valores faltantes


,faltantes,porcentaje
_BMI5,40535,9.35
INCOME3,8075,1.86
EMPLOY1,2968,0.68
EDUCA,9,0.00
GENHLTH,4,0.00
ADDEPEV3,3,0.00
EXERANY2,2,0.00


2. Tipos de datos
float64    13
Name: count, dtype: int64


3. Filas duplicadas (en las 13 variables seleccionadas): 252


In [33]:
# 4. Códigos especiales: en el BRFSS, 7/9 (o 77/99, 777/999) significan "no sabe" o "no responde".
#    Pandas los trata como números válidos, así que pueden distorsionar promedios.
codigos_especiales = {
    "EXERANY2": [7, 9], "GENHLTH": [7, 9], "ADDEPEV3": [7, 9], "EDUCA": [9],
    "EMPLOY1": [9], "INCOME3": [77, 99], "SLEPTIM1": [77, 99], "_TOTINDA": [9], "_AGEG5YR": [14],
}
filas = []
for col, codigos in codigos_especiales.items():
    if col in df_brfss.columns:
        n = df_brfss[col].isin(codigos).sum()
        filas.append({"variable": col, "códigos": codigos, "registros": n, "porcentaje": round(n / len(df_brfss) * 100, 2)})
pd.DataFrame(filas)

,variable,códigos,registros,porcentaje
0,EXERANY2,"[7, 9]",1249,0.29
1,GENHLTH,"[7, 9]",1258,0.29
2,ADDEPEV3,"[7, 9]",2584,0.60
3,EDUCA,[9],2316,0.53
4,EMPLOY1,[9],4713,1.09
5,INCOME3,"[77, 99]",78548,18.13
6,_TOTINDA,[9],1251,0.29
7,_AGEG5YR,[14],7779,1.80


In [34]:
# 5. Rangos sospechosos
if "SLEPTIM1" in df_brfss.columns:
    print("SLEPTIM1 (horas de sueño) mayores a 24 sin contar 77/99:",
          ((df_brfss["SLEPTIM1"] > 24) & ~df_brfss["SLEPTIM1"].isin([77, 99])).sum())
else:
    print("SLEPTIM1 (horas de sueño) no existe en el archivo BRFSS 2023: no se puede revisar su rango.")
print("_BMI5 tiene 2 decimales implícitos (ej. 2534 = 25.34). Rango actual:",
      df_brfss["_BMI5"].min(), "-", df_brfss["_BMI5"].max())
print("Valores únicos de EXERANY2:", sorted(df_brfss["EXERANY2"].dropna().unique()))

SLEPTIM1 (horas de sueño) no existe en el archivo BRFSS 2023: no se puede revisar su rango.
_BMI5 tiene 2 decimales implícitos (ej. 2534 = 25.34). Rango actual: 1202.0 - 9984.0
Valores únicos de EXERANY2: [np.float64(1.0), np.float64(2.0), np.float64(7.0), np.float64(9.0)]


### Dataset 2: Bogotá

In [35]:
print("1. Valores faltantes")
display(resumen_faltantes(df_bogota))

print("2. Columnas de texto que parecen números:", texto_que_parece_numero(df_bogota) or "ninguna")

print("3. Filas duplicadas:", df_bogota.duplicated().sum())

# 4. Porcentajes fuera del rango 0-100 (solo columnas que representan porcentajes o proporciones)
cols_pct = [c for c in df_bogota.columns if any(p in c.lower() for p in ("porc", "propor", "%"))]
for col in cols_pct:
    valores = pd.to_numeric(df_bogota[col].astype(str).str.replace(",", ".", regex=False), errors="coerce")
    fuera = ((valores < 0) | (valores > 100)).sum()
    print(f"   {col}: {fuera} valores fuera de 0-100; {valores.isna().sum()} no convertibles a número")
if not cols_pct:
    print("   No se identificaron columnas de porcentaje por su nombre; revisar manualmente.")

1. Valores faltantes


,faltantes,porcentaje


2. Columnas de texto que parecen números: {'Porcentaje': '100% numéricos'}
3. Filas duplicadas: 0
   Porcentaje: 0 valores fuera de 0-100; 0 no convertibles a número


In [36]:
# Valores únicos de las columnas de texto (para detectar categorías mal escritas)
for col in df_bogota.select_dtypes(include=["object"]).columns:
    print(col, "->", df_bogota[col].nunique(), "valores únicos")
    print("   ", df_bogota[col].dropna().unique()[:15])

Localidad -> 21 valores únicos


    <ArrowStringArray>
['Usaquén', 'Chapinero', 'Santa Fe', 'San Cristóbal', 'Usme', 'Tunjuelito', 'Bosa', 'Kennedy', 'Fontibón', 'Engativá', 'Suba', 'Barrios Unidos', 'Teusaquillo', 'Los Mártires', 'Antonio Nariño']
Length: 15, dtype: str
Percepción -> 4 valores únicos
    <ArrowStringArray>
['No practicó deporte ni tuvo actividad física en el mes', 'Menos de 1 vez a la semana', '1 a 2 veces por semana', '3 o más veces por semana']
Length: 4, dtype: str
Porcentaje -> 134 valores únicos
    <ArrowStringArray>
['36,7', '30,5', '46,1', '56,3', '57,5', '49,8', '55,2', '55,6', '46,6', '50,5', '45,2', '37,9', '35,8', '49,3', '46,0']
Length: 15, dtype: str


In [37]:
# 5. Cobertura temporal y coherencia de los porcentajes
print("Años disponibles:", sorted(df_bogota["Año"].unique().tolist()))
print("Registros por año:", df_bogota["Año"].value_counts().sort_index().to_dict())
print("Localidades (incluye el total 'Bogotá D.C.'):", df_bogota["Localidad"].nunique())
porcentaje_num = pd.to_numeric(df_bogota["Porcentaje"].str.replace(",", ".", regex=False), errors="coerce")
suma = porcentaje_num.groupby([df_bogota["Año"], df_bogota["Localidad"]]).sum().round(1)
print("Suma de las 4 categorías por localidad y año: mínimo", suma.min(), "- máximo", suma.max())
print("Combinaciones localidad-año cuya suma no es exactamente 100,0:", int((suma != 100).sum()), "de", len(suma))
display(suma[suma != 100].rename("suma").reset_index())


Años disponibles: [2017, 2021]
Registros por año: {2017: 84, 2021: 84}
Localidades (incluye el total 'Bogotá D.C.'): 21
Suma de las 4 categorías por localidad y año: mínimo 99.9 - máximo 102.6
Combinaciones localidad-año cuya suma no es exactamente 100,0: 18 de 42


,Año,Localidad,suma
0,2017,Antonio Nariño,99.9
1,2017,Barrios Unidos,100.1
2,2017,Ciudad Bolívar,100.1
3,2017,La Candelaria,100.1
4,2017,Los Mártires,100.1
5,2017,Rafael Uribe Uribe,100.1
6,2017,Santa Fe,100.1
7,2017,Tunjuelito,100.1
8,2017,Usme,100.1
9,2021,Antonio Nariño,100.1


### Dataset 3: Eurobarómetro 97.3

In [38]:
if df_eurobarometro is not None:
    print("1. Valores faltantes (20 columnas con más ausentes)")
    display(resumen_faltantes(df_eurobarometro).head(20))
    print("Columnas con faltantes:", (df_eurobarometro.isna().sum() > 0).sum(), "de", df_eurobarometro.shape[1])

    print("2. Tipos de datos")
    print(df_eurobarometro.dtypes.astype(str).value_counts())

    print("3. Filas duplicadas:", df_eurobarometro.duplicated().sum())
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")

1. Valores faltantes (20 columnas con más ausentes)


,faltantes,porcentaje
region_luxembourg,26067,98.11
region_cyprus,26066,98.11
region_malta,26065,98.10
region_finland,25571,96.24
region_lithuania,25567,96.23
region_denmark,25565,96.22
region_portugal,25564,96.22
region_austria,25564,96.22
region_spain,25563,96.21
region_croatia,25561,96.21


Columnas con faltantes: 163 de 501
2. Tipos de datos
int8       322
float64    166
int16        8
str          3
int32        2
Name: count, dtype: int64


3. Filas duplicadas: 0


In [39]:
if df_eurobarometro is not None:
    # 4. Códigos de "no sabe" / "no responde" / "no aplica".
    #    Como las respuestas se cargaron como números, se busca en las etiquetas de valor del archivo
    #    qué códigos significan DK, Refusal o Inap., y se cuentan los registros con esos códigos.
    #    (Buscar el texto "DK" en los datos daría falsos positivos: DK es también el código ISO de Dinamarca.)
    patron = r"(?i)\bDK\b|don.t know|refus|\binap\b|not asked"
    filas = []
    for col, mapa in etiquetas_valores.items():
        if col not in df_eurobarometro.columns:
            continue
        codigos = [c for c, texto in mapa.items() if re.search(patron, str(texto))]
        n = df_eurobarometro[col].isin(codigos).sum() if codigos else 0
        if n:
            filas.append({"variable": col, "etiqueta": etiquetas_variables.get(col, ""), "códigos": codigos,
                          "registros": n, "porcentaje": round(n / len(df_eurobarometro) * 100, 2)})
    codigos_euro = pd.DataFrame(filas).sort_values("registros", ascending=False)
    print(f"Variables con registros 'no sabe / no responde / no aplica': {len(codigos_euro)} de {df_eurobarometro.shape[1]}")
    display(codigos_euro.head(20))
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")


Variables con registros 'no sabe / no responde / no aplica': 167 de 501


,variable,etiqueta,códigos,registros,porcentaje
37,qa16_5,VIEW ABOUT: INDIA,[5],4778,17.98
77,qb20_3,GENDER EQUALITY IN SPORT - MORE ATTENTION FOR ...,[5],3350,12.61
36,qa16_4,VIEW ABOUT: TURKEY,[5],3193,12.02
147,d1r2,LEFT-RIGHT PLACEMENT - RECODED 5 CAT,"[6, 7]",3121,11.75
146,d1r1,LEFT-RIGHT PLACEMENT - RECODED 3 CAT,"[4, 5]",3121,11.75
145,d1,LEFT-RIGHT PLACEMENT,"[11, 12]",3121,11.75
67,qb13_3,SPORT - LOCAL AUTHORITY DOES NOT DO ENOUGH,[5],2769,10.42
75,qb20_1,GENDER EQUALITY IN SPORT - FEMALE ROLE MODELS ...,[5],2664,10.03
136,qc12_5,EU ENERGY POLICY - CAN CONTRIBUTE TO DEFEND ST...,[5],2658,10.00
66,qb13_2,SPORT - MANY OPPORTUNITIES BY LOCAL SPORTS CLUBS,[5],2568,9.67


In [40]:
if df_eurobarometro is not None:
    # 5. Módulo de deporte: frecuencia de práctica (qb1) y barreras declaradas (qb14_1 a qb14_13).
    #    Conteos sin ponderar: solo describen el archivo, no son estimaciones poblacionales.
    frecuencia = df_eurobarometro["qb1"].value_counts().sort_index()
    print("qb1 -", etiquetas_variables["qb1"])
    display(pd.DataFrame({
        "código": frecuencia.index,
        "respuesta": [etiquetas_valores["qb1"].get(c) for c in frecuencia.index],
        "registros": frecuencia.values,
        "porcentaje": (frecuencia.values / len(df_eurobarometro) * 100).round(2),
    }))
    barreras = [c for c in df_eurobarometro.columns if c.startswith("qb14_")]
    print("Códigos de qb14_1:", etiquetas_valores["qb14_1"])
    display(pd.DataFrame({
        "variable": barreras,
        "barrera": [etiquetas_variables[c] for c in barreras],
        "valores": [sorted(df_eurobarometro[c].dropna().unique().tolist()) for c in barreras],
        "mencionan (1)": [int((df_eurobarometro[c] == 1).sum()) for c in barreras],
        "porcentaje": [round((df_eurobarometro[c] == 1).mean() * 100, 2) for c in barreras],
    }))
else:
    print("Eurobarómetro no disponible: descárgalo según data/raw/README.md y vuelve a ejecutar el notebook.")


qb1 - SPORT ACTIVITY - FREQUENCY


,código,respuesta,registros,porcentaje
0,1,5 times a week or more,1876,7.06
1,2,3 to 4 times a week,3253,12.24
2,3,1 to 2 times a week,5047,19.00
3,4,1 to 3 times a month,1642,6.18
4,5,Less often,3563,13.41
5,6,Never,11161,42.01
6,7,DK (SPONT.),27,0.10


Códigos de qb14_1: {0: 'Not mentioned', 1: 'You do not have the time'}


,variable,barrera,valores,mencionan (1),porcentaje
0,qb14_1,NO SPORT ACTIVITY REASON: NO TIME,"[0, 1]",10729,40.38
1,qb14_2,NO SPORT ACTIVITY REASON: TOO EXPENSIVE,"[0, 1]",2583,9.72
2,qb14_3,NO SPORT ACTIVITY REASON: COMPETITIVITY,"[0, 1]",2538,9.55
3,qb14_4,NO SPORT ACTIVITY REASON: NO INFRASTRUCTURE,"[0, 1]",1605,6.04
4,qb14_5,NO SPORT ACTIVITY REASON: DISABILITY/ILLNESS,"[0, 1]",4435,16.69
5,qb14_6,NO SPORT ACTIVITY REASON: NO FRIENDS,"[0, 1]",1600,6.02
6,qb14_7,NO SPORT ACTIVITY REASON: FEEL DISCRIMINATED,"[0, 1]",345,1.30
7,qb14_8,NO SPORT ACTIVITY REASON: NO OPPORTUNITY W SAM...,"[0, 1]",667,2.51
8,qb14_9,NO SPORT ACTIVITY REASON: LACK MOTIVATION,"[0, 1]",6531,24.58
9,qb14_10,NO SPORT ACTIVITY REASON: RISK OF INJURIES,"[0, 1]",2497,9.40


## Tabla de hallazgos de calidad

| Dataset | Problema identificado | Evidencia encontrada | Posible impacto | Acción propuesta |
|---|---|---|---|---|
| BRFSS 2023 | Códigos de "no sabe / no responde" guardados como números | `INCOME3` 77/99: 78 548 registros (18,13 %); `_AGEG5YR` 14: 7 779 (1,80 %); `EMPLOY1` 9: 4 713 (1,09 %); `EXERANY2` 7/9: 1 249 (0,29 %) | Distorsionan promedios y porcentajes | Convertirlos a valores faltantes; en ingresos, crear la categoría "No informa" para no perder el 18 % de los registros |
| BRFSS 2023 | Valores faltantes | `_BMI5`: 40 535 (9,35 %); `INCOME3`: 8 075 (1,86 %); `EMPLOY1`: 2 968 (0,68 %) | Menos registros para analizar el IMC | Conservar y analizar el IMC solo con los casos disponibles; no imputar |
| BRFSS 2023 | Tipos de datos inadecuados | Las 13 columnas son `float64`, incluidas las categóricas | Interpretaciones erróneas y más memoria | Convertir a categorías con etiquetas y a enteros |
| BRFSS 2023 | Escala de `_BMI5` | Rango de 1 202 a 9 984 | IMC 100 veces mayor al real | Dividir entre 100 |
| BRFSS 2023 | Filas idénticas | 252 duplicados en las 13 variables, sin identificador | Eliminarlas borraría personas reales | Conservarlas y crear un identificador por registro |
| BRFSS 2023 | Variable ausente | `SLEPTIM1` no existe en 2023 | No se puede analizar el sueño | Usar salud general, IMC y depresión como factores de salud |
| Bogotá | Porcentaje como texto | `Porcentaje` es `str` con coma decimal (100 % convertible) | No se pueden hacer cálculos | Convertir a número decimal |
| Bogotá | Total mezclado con localidades | "Bogotá D.C." aparece como una localidad más | Doble conteo en rankings y promedios | Marcar y separar la fila total |
| Bogotá | Datos agregados de 2 años | 168 filas de porcentajes (2017 y 2021) | No permite análisis por persona | Usarlo como contexto local |
| Eurobarómetro | Columnas que no aplican al proyecto | 501 columnas de todos los módulos | Más memoria y confusión | Seleccionar el módulo de deporte y las variables sociodemográficas |
| Eurobarómetro | Respuestas codificadas | 322 columnas `int8`; etiquetas duplicadas en `d9b` | Datos ilegibles sin el libro de códigos | Recodificar con los metadatos de `pyreadstat` |
| Eurobarómetro | Faltantes estructurales | 163 de 501 columnas con faltantes; las 20 con más ausentes son `region_*` (96,12 % a 98,11 %) | No son errores: solo aplican a un país | Excluir esas columnas |
| Eurobarómetro | Falso positivo al buscar "DK" | "DK" es también el código ISO de Dinamarca en `isocntry`; con las etiquetas de valor, 167 de 501 variables tienen registros "no sabe / no responde / no aplica" | Contar mal las respuestas "no sabe" | Identificar esos códigos desde las etiquetas de valor |

## Conclusiones

Los tres datasets se cargan correctamente y sirven para el proyecto, pero **ninguno puede usarse directamente**: el BRFSS necesita
recodificar sus códigos especiales, corregir tipos y escalar el IMC; el archivo de Bogotá necesita convertir los porcentajes a números y separar el total;
y el Eurobarómetro necesita reducirse al módulo de deporte y traducir sus códigos con los metadatos. La principal limitación es que la
variable de sueño no existe en el BRFSS 2023 y que las fuentes con microdatos (BRFSS y Eurobarómetro) no son colombianas. Las acciones de limpieza
propuestas se implementan en `notebooks/data_preparation.ipynb`.